# Introspect-AI — LODO training on a Colab runtime

Trains the GAT+LSTM `WorldModel` (`backend/train.py`) with leave-one-day-out
cross-validation over the five CIC-IDS-2017 day slices.

**Read this first — it changes how you use the notebook:**

The Colab runtime is a *remote VM*. It cannot see the files on your laptop, even
though this notebook is open in your local VS Code. Code and data get there via
Google Drive, which is what steps 2–4 do.

**About the GPU.** Measured on the real slices: ~68 ms per training sequence on
CPU, with graphs averaging 115 nodes / 133 edges. The model runs one sequence at
a time (batch size 1) over up to 20 tiny graphs, so a T4 spends more time
launching kernels than computing. A GPU will not meaningfully speed this up —
batching the graphs would, and that is a code change, not a runtime change.
Pick a **CPU** runtime unless you are measuring the difference yourself.

What Colab *does* buy you: a machine that runs for hours without tying up your
laptop, and the option to run several configurations in parallel sessions.

**Expected wall clock:** ~1000 windows total, ~800 training sequences per fold,
5 folds x 30 epochs. That is roughly 55 s/epoch on 6 threads, so ~30 min/fold
and **~2.5 h for the full LODO run** — longer on Colab's 2 vCPU. Step 7 runs it
in the background and checkpoints every fold so a disconnect costs one fold, not
the whole run.

## 1. Inspect the runtime

Confirms what hardware you actually got, and that torch/sklearn/pydantic are
present. Nothing is installed yet.

In [ ]:
import os, sys, platform, subprocess

print("python  :", sys.version.split()[0], "|", platform.platform())
print("cpu     :", os.cpu_count(), "logical cores")

try:
    with open("/proc/meminfo") as f:
        total_kb = int(f.readline().split()[1])
    print("ram     : %.1f GB" % (total_kb / 1024 / 1024))
except FileNotFoundError:
    pass

import torch, sklearn, numpy, pydantic
print("torch   :", torch.__version__, "| threads:", torch.get_num_threads())
print("sklearn :", sklearn.__version__)
print("numpy   :", numpy.__version__)
print("pydantic:", pydantic.__version__)
print("cuda    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "unavailable (expected on a CPU runtime)")

## 2. Mount Google Drive

In the VS Code extension this is also available as the command
**`Colab: Mount Google Drive to Server...`** from the command palette. The cell
below does the same thing and works identically in browser Colab.

Drive is where the durable artifacts live: the uploaded archives, and every
checkpoint and metrics file the run produces. Anything written only to
`/content` disappears when the runtime is recycled.

In [ ]:
from google.colab import drive  # type: ignore[import-not-found]  # provided by the Colab runtime only
drive.mount("/content/drive")

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/introspect-ai")
CODE_ZIP   = DRIVE_ROOT / "introspect_code.zip"
DATA_ZIP   = DRIVE_ROOT / "introspect_data.zip"
RUN_DIR    = DRIVE_ROOT / "runs"          # checkpoints + lodo_results.json land here

RUN_DIR.mkdir(parents=True, exist_ok=True)

for p in (CODE_ZIP, DATA_ZIP):
    if p.exists():
        print("found  %-24s %6.1f MB" % (p.name, p.stat().st_size / 1e6))
    else:
        print("MISSING", p, "<- run scripts/pack_for_colab.ps1 locally and upload it")

## 3. Unpack code and data onto the runtime

Both archives are extracted to local disk (`/content/introspect`), not read from
Drive directly — Drive's FUSE mount is slow for the many small reads the CSV
parser does.

Re-run this cell after every code change. It wipes `backend/app` and `train.py`
so a stale module can never shadow the new one, but leaves `data/raw` alone so
you are not re-extracting 300 MB each time.

In [ ]:
import shutil, zipfile, time
from pathlib import Path

WORK    = Path("/content/introspect")
BACKEND = WORK / "backend"
RAW     = BACKEND / "data" / "raw"

# --- code: always refreshed -------------------------------------------------
for stale in (BACKEND / "app", BACKEND / "train.py"):
    if stale.is_dir():
        shutil.rmtree(stale)
    elif stale.exists():
        stale.unlink()

WORK.mkdir(parents=True, exist_ok=True)
t0 = time.time()
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(WORK)
print("code extracted in %.1fs" % (time.time() - t0))

# --- data: extracted once ---------------------------------------------------
SLICES = [
    "monday_plus_slice.csv",
    "tuesday_plus_slice.csv",
    "wednesday_plus_slice.csv",
    "thursday_plus_slice.csv",
    "friday_plus_slice_mixed.csv",
]

RAW.mkdir(parents=True, exist_ok=True)
missing = [s for s in SLICES if not (RAW / s).exists()]

if missing:
    print("extracting %d slice(s): %s" % (len(missing), ", ".join(missing)))
    t0 = time.time()
    with zipfile.ZipFile(DATA_ZIP) as z:
        for name in z.namelist():
            base = Path(name).name
            if base in missing:
                with z.open(name) as src, open(RAW / base, "wb") as dst:
                    shutil.copyfileobj(src, dst, length=8 << 20)
    print("data extracted in %.1fs" % (time.time() - t0))
else:
    print("all slices already present")

for s in SLICES:
    p = RAW / s
    print("  %-32s %8.1f MB" % (s, p.stat().st_size / 1e6) if p.exists() else "  MISSING " + s)

## 4. Dependencies

Colab already ships torch, numpy, scikit-learn and pydantic v2, which is
everything `train.py` imports. **Do not `pip install -r requirements.txt`** —
that file pins `torch==2.4.1+cpu` for Windows and would replace Colab's build
with a slower one (or break the CUDA stack) for no benefit. `fastapi` and
`pandas` are listed there for the API and ingestion work, but the training path
does not import them.

The cell below only repairs a pydantic that is too old to be compatible.

In [ ]:
import subprocess, sys
import pydantic

major, minor = (int(x) for x in pydantic.__version__.split(".")[:2])
if (major, minor) < (2, 9):
    print("pydantic", pydantic.__version__, "is older than 2.9 - upgrading")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic>=2.9,<3"], check=True)
    print("RESTART the runtime, then re-run from step 2.")
else:
    print("pydantic", pydantic.__version__, "- nothing to install")


## 5. Sanity checks

Two gates before committing hours to the full run.

1. **`pytest`** — 123 tests covering the parser, extractor, graph builder, GAT,
   LSTM and `WorldModel`. Runs in about 3 seconds and catches a bad upload or a
   version mismatch immediately.
2. **A 1-epoch LODO run on truncated slices** — takes the first 3000 rows of each
   day into a throwaway directory and runs the real `train.py` end to end: the
   CIC-IDS parser, windowing, all five folds, per-fold checkpointing and the
   pooled aggregate. About a minute, and it exercises exactly the code path the
   long run uses.

> `train.py --smoke` is **not** used here. The mock dataset resolves to a single
> day, so the LODO hold-out leaves an empty training pool and the run raises
> `ValueError: not enough values to unpack`. That is a pre-existing gap in the
> in-progress LODO refactor, unrelated to Colab.

`cwd` must be `backend/` throughout: `train.py` does `from app import config`,
and its data paths are relative.

In [ ]:
import subprocess, sys, shutil, itertools
from pathlib import Path

# --- gate 1: unit tests -----------------------------------------------------
proc = subprocess.run([sys.executable, "-m", "pytest", "-q"],
                      cwd=BACKEND, capture_output=True, text=True)
print("\n".join((proc.stdout + proc.stderr).strip().splitlines()[-6:]))
assert proc.returncode == 0, "pytest failed - stop here and fix it"

# --- gate 2: 1-epoch LODO on truncated slices -------------------------------
MINI_DATA = Path("/content/mini/data")
MINI_CKPT = Path("/content/mini/ckpt")
for d in (MINI_DATA, MINI_CKPT):
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True, exist_ok=True)

for s in SLICES:
    with open(RAW / s, encoding="utf-8-sig") as src, open(MINI_DATA / s, "w", encoding="utf-8") as dst:
        dst.writelines(itertools.islice(src, 3001))   # header + 3000 rows

proc = subprocess.run(
    [sys.executable, "train.py",
     "--epochs", "1",
     "--data-dir", str(MINI_DATA),
     "--save-path", str(MINI_CKPT),
     "--device", "cpu"],
    cwd=BACKEND, capture_output=True, text=True,
)
out = proc.stdout + proc.stderr
for line in out.splitlines():
    if any(k in line for k in ("Fold", "Pooled", "Saved", "Traceback", "device")):
        print(line)
print("\nexit code:", proc.returncode)
assert proc.returncode == 0, out[-3000:]

print("fold artifacts:", sorted(p.name for p in MINI_CKPT.glob("lodo_fold_*.pt")))
print("\nBoth gates passed - safe to launch the full run.")

## 6. Confirm the label distribution

Cheap guard against a silently truncated upload. The counts below should match
what you see locally:

| slice | rows | attack rows |
|---|---|---|
| monday | 50 000 | 0 — all BENIGN |
| tuesday | 100 000 | ~2 957 Patator |
| wednesday | 260 000 | ~163 500 DoS |
| thursday | 100 000 | ~24 400 Infiltration / Web |
| friday (mixed) | 78 000 | ~37 700 DDoS / Botnet / Portscan |

Monday being 100 % benign is expected and handled — that fold's test set is
single-class, so its ROC-AUC is reported as `N/A` and only specificity is
meaningful for it.

In [ ]:
import csv
from collections import Counter

for s in SLICES:
    with open(RAW / s, newline="", encoding="utf-8-sig") as f:
        r = csv.DictReader(f)
        field = next((c for c in r.fieldnames if c.strip().lower() == "label"), None)
        counts = Counter(row[field].strip() for row in r)
    total  = sum(counts.values())
    benign = counts.get("BENIGN", 0) + counts.get("Benign", 0)
    top    = ", ".join("%s=%d" % kv for kv in counts.most_common(4) if kv[0].upper() != "BENIGN")
    print("%-32s rows=%-7d attack=%-7d %s" % (s, total, total - benign, top or "(none)"))

## 7. Launch the LODO run

Started with `nohup` so it is detached from this notebook's kernel: losing the
websocket to VS Code no longer kills training. Logs stream to
`/content/train.log`; step 8 tails it.

Each fold writes `lodo_fold_<day>.pt` and updates `lodo_results.json` **directly
into Drive** as soon as it finishes, so an interrupted run loses at most the
in-progress fold. Step 9 resumes from there.

`--epochs 30` is passed explicitly to match the value the script used when
`EPOCHS` was hardcoded — change it here, not in the source.

In [ ]:
import subprocess, shlex, os

LOG = "/content/train.log"

cmd = [
    "python", "train.py",
    "--epochs", "30",
    "--save-path", str(RUN_DIR),
    "--device", "cpu",          # 'cuda' is supported but see the note at the top
    "--resume-folds",           # harmless on a fresh run; required after a disconnect
]

with open(LOG, "ab") as log:
    log.write(b"\n==== launch: " + subprocess.check_output(["date", "-u"]) + b"====\n")

proc = subprocess.Popen(
    ["nohup"] + cmd,
    cwd=BACKEND,
    stdout=open(LOG, "ab"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setpgrp,
)
print("pid", proc.pid, "->", LOG)
print("cmd:", " ".join(shlex.quote(c) for c in cmd))

## 8. Follow progress

Re-run this cell whenever you want a fresh view; it is read-only and safe to run
while training continues. It also copies the log to Drive so the run is
diagnosable even if the runtime is gone by the time you come back.

In [ ]:
import shutil, subprocess

LOG = "/content/train.log"   # redefined so this cell works after a reconnect

alive = subprocess.run(["pgrep", "-f", "train.py"], capture_output=True, text=True).stdout.split()
print("train.py running:", bool(alive), "(pids: %s)" % (", ".join(alive) or "none"))

with open(LOG) as f:
    lines = f.read().splitlines()

print("--- %d log lines, last 30 ---" % len(lines))
print("\n".join(lines[-30:]))

shutil.copy(LOG, RUN_DIR / "train.log")

done = sorted(p.stem.replace("lodo_fold_", "") for p in RUN_DIR.glob("lodo_fold_*.pt"))
print("\nfolds checkpointed to Drive:", done or "none yet")

## 9. After a disconnect

A Colab runtime can be recycled for idleness (~90 min) or at the session cap
(~12 h on the free tier), and the notebook's own connection can drop without the
runtime dying. Two different situations:

- **Runtime still alive, notebook reconnected** — training never stopped. Just
  re-run step 8.
- **Runtime was recycled** — re-run steps 2, 3 and 7. Because step 7 passes
  `--resume-folds`, the finished folds are read back from
  `lodo_results.json` in Drive and skipped; training picks up at the first
  unfinished fold.

The one cost of a recycle is re-parsing the CSVs (~25 s for all five slices at
the measured 26 500 rows/s), which happens before the fold loop regardless.

## 10. Results

`lodo_results.json` holds one entry per completed fold. Precision is
deliberately *not* averaged across folds — the class priors differ per day, so a
mean would be meaningless. Specificity and recall are pooled over raw counts in
the training log's `POOLED AGGREGATE` section.

In [ ]:
import json

results_file = RUN_DIR / "lodo_results.json"
if not results_file.exists():
    print("no folds finished yet")
else:
    results = json.loads(results_file.read_text())
    hdr = "%-11s %5s %5s %5s %5s  %-8s %-8s %-8s %s"
    print(hdr % ("fold", "TN", "FP", "FN", "TP", "recall", "prec", "f1", "roc_auc"))
    for day, m in results.items():
        (tn, fp), (fn, tp) = m["attack_confusion_matrix"]
        auc = m.get("attack_roc_auc")
        auc = "%.4f" % auc if isinstance(auc, (int, float)) else str(auc)
        print(hdr % (day, tn, fp, fn, tp,
                     "%.4f" % m["attack_recall"], "%.4f" % m["attack_precision"],
                     "%.4f" % m["attack_f1"], auc))

    tn_t = sum(m["attack_confusion_matrix"][0][0] for m in results.values())
    fp_t = sum(m["attack_confusion_matrix"][0][1] for m in results.values())
    fn_t = sum(m["attack_confusion_matrix"][1][0] for m in results.values())
    tp_t = sum(m["attack_confusion_matrix"][1][1] for m in results.values())
    print("\npooled specificity: %.4f (n=%d)" % (tn_t / max(1, tn_t + fp_t), tn_t + fp_t))
    print("pooled recall     : %.4f (n=%d)" % (tp_t / max(1, tp_t + fn_t), tp_t + fn_t))

## 11. Bring artifacts home

Everything already sits in `MyDrive/introspect-ai/runs/`, so the Drive desktop
client or the web UI is the simplest route. Use the cell below only if you want
a single file in your browser's downloads.

Do **not** commit these to git — `backend/checkpoints/` is gitignored on purpose.

In [ ]:
import os, shutil
from google.colab import files  # type: ignore[import-not-found]  # provided by the Colab runtime only

bundle = shutil.make_archive("/content/lodo_run", "zip", RUN_DIR)
print("%.1f MB" % (os.path.getsize(bundle) / 1e6))
files.download(bundle)

## Known limits of this setup

- **No GPU speedup without batching.** Measured above: the bottleneck is Python
  overhead and tiny per-graph kernels, not FLOPs. Batching the windows of a
  sequence into one disjoint-union graph is the change that would pay off, and it
  would help on CPU too.
- **Colab's 2 vCPU can be slower than your laptop's 6 threads.** Colab's value
  here is unattended duration and parallel experiments, not raw speed.
- **`run_pipeline` returns only the last fold's model.** LODO is an evaluation
  protocol; no single deployment model comes out of it. To produce one, train on
  all five days without a hold-out.
- **The MITRE head is currently not learning.** `train_one_epoch` multiplies its
  loss by `0.0` (a deliberate diagnostic in the current working tree). MITRE
  metrics from this run reflect an untrained head.
- **The DoS labels rely on the fallback mapping.** `LABEL_TO_MITRE` has no entry
  for `DoS Hulk`/`Slowloris`/`GoldenEye`/`Slowhttptest`, so they fall through to
  `MitreStage.IMPACT`. That is a reasonable mapping, but it is implicit — worth an
  explicit entry if MITRE accuracy starts mattering.
- **`train.py --smoke` is broken under LODO.** The mock dataset is a single day,
  so holding it out empties the training pool. Use `pytest` plus the truncated-slice
  run in step 5 as the sanity gate until the smoke path is reworked.
